## TESTING THE SIMULATION WITH REAL DATA

In this section we will compare the results of our portfolio simulation with the real results we would have got with the real stock market data.

In [1]:
import sys
import os

parent_dir = os.path.abspath('..')

if parent_dir not in sys.path:
	sys.path.append(parent_dir)

In [2]:
from src.simulation import simulate_portfolio_returns
import numpy as np
import pandas as pd
import yfinance as yf

First we download the real stock data of the next 30 trading days of our raw data to make the test. 

In [9]:
days_30 = yf.download(['SAN', 'ITX.MC', 'AIR.PA', 'SIE.DE', 'IBE.MC'], start="2025-01-01", end="2025-01-31", interval='1d', group_by='ticker')

df = pd.DataFrame(days_30)
df = df.filter(like='Close', axis=1) # Keep only the closing prices

[*********************100%***********************]  5 of 5 completed


### Simulation

In [10]:
returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, header=0)
cov_matrix = pd.read_csv('../data/processed/covariance_matrix.csv', index_col=0, header=0)
cov_matrix = cov_matrix/252

np.random.seed(2026) # For reproducibility

T = 30 # Time horizon in days
M = 10000 # Number of simulations
P = 100000 # Portfolio value in euros - Equally weighted portfolio between the 5 stoks

portfolio_final_values = simulate_portfolio_returns(returns, cov_matrix, T, M, P)

profit = portfolio_final_values - P # Profit distribution

var_95 = np.percentile(profit, 5) # 5th percentile of the profit distribution

print(var_95)

-9086.383566351113


### Real Data

In [11]:
df

Ticker,AIR.PA,SAN,SIE.DE,IBE.MC,ITX.MC
Price,Close,Close,Close,Close,Close
Date,,,,,
2025-01-02,153.777924,4.331857,180.949371,12.584102,47.785370
2025-01-03,152.279892,4.360995,178.710480,12.741634,47.462502
2025-01-06,153.297791,4.496973,184.738297,12.792602,48.355152
2025-01-07,152.107040,4.555248,185.178406,12.704567,48.279179
2025-01-08,152.279892,4.526111,187.187683,12.681401,48.127239
2025-01-09,150.359360,NaN,188.106232,12.732368,48.526085
2025-01-10,151.300415,4.487259,186.096954,12.422280,47.614441
2025-01-13,149.264648,4.516398,184.833954,12.379869,46.446396


In [ ]:
first_day = df.iloc[0]
day_30 = df.iloc[-1]

prices = day_30.values/first_day.values

prices = prices*P/returns.shape[1]

final_portfolio_value = np.sum(prices)

profit = final_portfolio_value - P

print(profit)

7209.107142238747
